## 1. 피처(X)와 타겟(y) 분리 및 전처리

- X, y 분리: X = df.drop(columns=['타겟컬럼']), y = df['타겟컬럼']

- 스케일링 (정규화/표준화):

  - 거리 기반 모델(KNN, SVM, K-Means 등)은 스케일링이 필수.

  - MinMaxScaler (0~1로 변환), StandardScaler (평균 0, 분산 1로 변환).

    - 주의! Train 데이터는 fit_transform(), Test 데이터는 transform() 적용.

  - 원-핫 인코딩 (One-Hot Encoding): pd.get_dummies(df, columns=['범주형컬럼'])

## 2. 머신러닝 모델링 파이프라인 (공통 문법)

- 1단계. 객체 생성: model = RandomForestRegressor(파라미터)

- 2단계. 모델 학습: model.fit(X_train, y_train)

- 3단계. 예측 수행: pred = model.predict(X_test)

## 3. 지도학습 주요 모델

- 회귀(Regression):
  - LinearRegression
  - Ridge
  - RandomForestRegressor
  - DecisionTreeRegressor

- 분류(Classification):
  - SVC
  - KNeighborsClassifier
  - DecisionTreeClassifier (특성 중요도는 model.feature_importances_로 확인)

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# [세트 1: 팁 예측 (회귀)] - 데이터: sns.load_dataset('tips')
def predict_tip(df: pd.DataFrame, df_test: pd.DataFrame):
    # 1. df에서 'tip' 컬럼을 종속변수 y, 나머지 컬럼을 독립변수 X로 분리
    X = df.drop(columns=['tip'])
    y = df['tip']

    # 2. RandomForestRegressor(max_depth=3, n_estimators=100) 객체 생성 후 X, y로 학습
    model = RandomForestRegressor(max_depth=3, n_estimators=100, random_state=42)
    model.fit(X, y)

    # 3. 학습된 모델로 df_test를 예측하여 pred 변수에 저장 후 반환
    pred = model.predict(df_test)
    return pred

tips_df = sns.load_dataset('tips')

# 범주형 데이터를 숫자형으로 매핑
tips_df['sex'] = tips_df['sex'].map({'Female': 0, 'Male': 1})
tips_df['smoker'] = tips_df['smoker'].map({'No': 0, 'Yes': 1})
tips_df['day'] = tips_df['day'].map({'Thur': 0, 'Fri': 1, 'Sat': 2, 'Sun': 3})
tips_df['time'] = tips_df['time'].map({'Lunch': 0, 'Dinner': 1})

display(tips_df.head())

df_train, df_test = train_test_split(tips_df, test_size=0.2, random_state=42)

# 예측 수행 (테스트 데이터에서도 y값인 'tip'은 제외하고 전달)
result = predict_tip(df_train, df_test.drop(columns=['tip']))
print(f"예측 결과 샘플: {result[:5]}")

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split

# [세트 2: 펭귄 몸무게 예측 (선형 회귀)] - 데이터: sns.load_dataset('penguins')
def predict_penguin_weight(df: pd.DataFrame, df_test: pd.DataFrame):
    # 1. 'body_mass_g' 컬럼을 y, 나머지를 X로 분리
    X = df.drop(columns=['body_mass_g'])
    y = df['body_mass_g']

    # 2. LinearRegression 모델 객체 lr_model 생성
    lr_model = LinearRegression()
    lr_model.fit(X, y)

    # 3. Ridge(alpha=1.0) 모델 객체 ridge_model 생성
    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(X, y)

    # 4. 각 모델을 X, y로 학습 후 df_test 예측 결과 lr_pred, ridge_pred 반환
    lr_pred = lr_model.predict(df_test)
    ridge_pred = ridge_model.predict(df_test)

    return lr_pred, ridge_pred

# 데이터 로드 및 전처리
penguins = sns.load_dataset('penguins')
# 단순화를 위해 결측치 제거 및 범주형 변수 원-핫 인코딩
penguins = penguins.dropna()
penguins = pd.get_dummies(penguins, columns=['species', 'island', 'sex'], drop_first=True)

# 학습/테스트 데이터 분리
train_df, test_df = train_test_split(penguins, test_size=0.2, random_state=42)

# 예측 수행
lr_results, ridge_results = predict_penguin_weight(train_df, test_df.drop(columns=['body_mass_g']))

print(f"LinearRegression 예측값(5개): {lr_results[:5]}")
print(f"Ridge 예측값(5개): {ridge_results[:5]}")

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# [세트 3: 타이타닉 생존 예측 (분류 & 스케일링)] - 데이터: sns.load_dataset('titanic')
def predict_survival(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: pd.Series):
    # 1. MinMaxScaler 객체를 만들어 X_train, X_test 정규화
    scaler = MinMaxScaler()
    # Train 데이터는 fit_transform, Test 데이터는 transform 적용
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 2. SVC() 모델 객체를 만들어 정규화된 X_train, y_train으로 학습
    model = SVC(random_state=42)
    model.fit(X_train_scaled, y_train)

    # 3. 정규화된 X_test 예측값을 pred에 저장 후 (모델, pred) 반환
    pred = model.predict(X_test_scaled)
    return model, pred

# 데이터 로드 및 전처리
titanic = sns.load_dataset('titanic')
# 4. 분석에 사용할 수치형/범주형 컬럼 선택 및 결측치 처리
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
titanic = titanic[cols].dropna()

# 범주형 변수 인코딩
titanic['sex'] = titanic['sex'].map({'male': 0, 'female': 1})

# 5. X, y 분리
X = titanic.drop(columns=['survived'])
y = titanic['survived']

# 6. 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 함수 실행
model, predictions = predict_survival(X_train, X_test, y_train)

print(f"모델 정확도: {accuracy_score(y_test, predictions):.4f}")
print(f"예측 결과 샘플: {predictions[:10]}")

## 4. 비지도학습 주요 모델

- 군집화(Clustering): KMeans(n_clusters=K) (답(y)이 없는 데이터(X)를 K개의 그룹으로 묶음)

- 차원 축소(PCA): PCA(n_components=N) (데이터의 분산을 최대한 보존하며 고차원 데이터를 N차원으로 축소)

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# [세트 1: 붓꽃 데이터 군집화] - 데이터: sns.load_dataset('iris')
def cluster_iris(df: pd.DataFrame):
    # 1.'species' 컬럼을 제외한 데이터 선택
    X = df.drop(columns=['species'])

    # 2. KMeans(n_clusters=3, random_state=42) 모델 생성
    kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')

    # 3. 'species' 컬럼을 제외한 데이터로 KMeans 학습 후 결과 반환
    kmeans.fit(X)
    return kmeans.labels_

def reduce_dim(df: pd.DataFrame):
    # 4. 'species' 컬럼을 제외한 데이터 선택
    X = df.drop(columns=['species'])

    # 5. PCA(n_components=2) 객체 생성
    pca = PCA(n_components=2)

    # 6. 'species' 제외 데이터 변환 결과를 pca_transformed 에 저장 후 반환
    pca_transformed = pca.fit_transform(X)
    return pca_transformed

# 실행 및 결과 확인
iris = sns.load_dataset('iris')
labels = cluster_iris(iris)
pca_res = reduce_dim(iris)

print(f"군집 결과 샘플 (10개): {labels[:10]}")
print(f"PCA 변환 결과 샘플 (2개):\n{pca_res[:2]}")

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

# [세트 2: 데이터 병합 및 의사결정나무 회귀]
def merge_and_predict(df1: pd.DataFrame, df2: pd.DataFrame, df_test: pd.DataFrame):
    # 1. df1, df2를 위아래로 합침 (인덱스 초기화)
    df_merged = pd.concat([df1, df2], ignore_index=True)

    # 2. 특정 불필요 컬럼(예: 'id') 제거 후 'target'을 y, 나머지를 X로 분리
    X = df_merged.drop(columns=['target'])
    y = df_merged['target']

    # 3. DecisionTreeRegressor()로 학습 후 df_test 예측값 반환
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X, y)

    pred = model.predict(df_test)
    return pred

# 예시 데이터 생성 및 실행 코드
data1 = {'feature1': [10, 20], 'target': [100, 200]}
data2 = {'feature1': [30, 40], 'target': [300, 400]}
test_data = {'feature1': [50]}

df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)
df_test = pd.DataFrame(test_data)

predictions = merge_and_predict(df1, df2, df_test)
print(f"예측 결과: {predictions}")

In [ ]:
import pandas as pd
import seaborn as sns

# [세트 3: 파생변수 생성 및 원-핫 인코딩]
def transform_features(df: pd.DataFrame):
    # 1. 'size' 컬럼명을 'party_size'로 변경
    df = df.rename(columns={'size': 'party_size'})

    # 2. 'total_bill'을 'low(0~15)', 'mid(15~30)', 'high(30~)' 범주로 변환하는 bin_bill 함수 작성 및 적용
    def bin_bill(bill):
        if bill < 15:
            return 'low'
        elif bill < 30:
            return 'mid'
        else:
            return 'high'

    df['total_bill_cat'] = df['total_bill'].apply(bin_bill)

    # 3. pd.get_dummies()를 사용해 변환된 'total_bill_cat' 컬럼 원-핫 인코딩 후 반환
    df_transformed = pd.get_dummies(df, columns=['total_bill_cat'])
    return df_transformed

# 실행 및 확인
tips = sns.load_dataset('tips')
transformed_tips = transform_features(tips)
display(transformed_tips.head())

## 5. 클래스 불균형 처리 (Imbalanced Data):

  - from imblearn.over_sampling import SMOTE (오버샘플링: 소수 클래스 증식)

  - from imblearn.under_sampling import RandomUnderSampler (언더샘플링: 다수 클래스 축소)

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# [세트 1: KNN과 StandardScaler 적용]
def train_knn(df: pd.DataFrame):
    # 1. 특정 이진 분류 컬럼을 0, 1로 매핑 (예: 'sex' -> male:0, female:1)
    if 'sex' in df.columns:
        df['sex'] = df['sex'].map({'male': 0, 'female': 1})

    # 결측치 처리 (Titanic 데이터셋 기준)
    df = df.dropna()

    # 종속 변수 분리 및 나머지 특성 X에 대해 StandardScaler 적용 ('survived'컬럼이 종속변수)
    X = df.drop(columns=['survived'])
    y = df['survived']

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 3. train_test_split(test_size=0.2, stratify=y, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    # 4. KNeighborsClassifier(n_neighbors=3) 학습 및 예측 정확도 반환
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    return accuracy

# 실행 및 확인 (Titanic 데이터 예시)
titanic = sns.load_dataset('titanic')[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']]
acc = train_knn(titanic)
print(f"KNN 모델 정확도: {acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

# [세트 2: 의사결정나무 시각화]
def tree_feature_importance(X_train: pd.DataFrame, y_train: pd.Series):
    # 1. DecisionTreeClassifier(random_state=42) 모델 학습
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)

    # 2. model.feature_importances_ 를 추출
    importances = model.feature_importances_
    feature_names = X_train.columns

    # 3. plt.barh()를 사용하여 특성 중요도 수평 막대 그래프 시각화 코드 작성
    plt.figure(figsize=(10, 6))
    plt.barh(feature_names, importances)
    plt.show()

# 실행 및 확인 (Titanic 데이터 활용)
tree_feature_importance(X_train, y_train)

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# [세트 3: SMOTE 및 UnderSampling]
def apply_sampling(X, y):
    # 1. RandomUnderSampler(random_state=42) 적용하여 X_rus, y_rus 생성
    rus = RandomUnderSampler(random_state=42)
    X_rus, y_rus = rus.fit_resample(X, y)

    return X_rus, y_rus

def apply_sampling_smote(X, y):
    # 2. SMOTE(random_state=42) 적용하여 X_smote, y_smote 생성
    smote = SMOTE(random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    return X_smote, y_smote

# 실행 및 확인 (Titanic 데이터의 survived 클래스 비율 확인)
print("원본 데이터 클래스 분포:\n", y.value_counts())

X_r, y_r = apply_sampling(X, y)
X_s, y_s = apply_sampling_smote(X, y)

print("\nSMOTE 적용 후 분포:\n", y_s.value_counts())
print("UnderSampling 적용 후 분포:\n", y_r.value_counts())